# 03c · CDD-CESM eligibility count + external test on DMID

**Runtime: CPU** (Runtime → Change runtime type → CPU). Read-only for the training runs; it never writes into `p2_cbis_runs_e200`.

1. Counts how many CDD-CESM low-energy (DM) images are usable mass segmentation cases (annotation files only; no images needed).
2. Tests every **finished** CBIS run (`summary.json` + `best.weights.h5`) on the DMID mass crops (`roi` mask set):
   normal prediction + severed bottleneck (constant / swap, bank built from that run's own CBIS training images).
   It also re-scores each run on its own CBIS test fold — this must match the training summary (weights loaded correctly).
3. Results: `results/p2_external_dmid_roi/<run>/per_image.csv` + `summary.json`, and `results/p2_external_dmid_roi_table.csv`.

Resumable: re-running skips runs already tested; re-run later to add runs that finished training since.


In [ ]:
# 1 · Drive, paths, imports
from google.colab import drive
drive.mount('/content/drive')
import os, sys, glob, json, time
import numpy as np, pandas as pd

PROJECT  = '/content/drive/MyDrive/quantum-placement'
MASK_SET = 'roi'                                   # chosen from the 03b QC (pixel masks = whole breast, not usable)
CBIS_DIR = f'{PROJECT}/data/processed/cbis_mass_256'
DMID_DIR = f'{PROJECT}/data/external/dmid_mass_256/{MASK_SET}'
CDD_DIR  = f'{PROJECT}/data/external/cdd_cesm'
RUNS_DIR = f'{PROJECT}/results/p2_cbis_runs_e200'       # read only
OUT_DIR  = f'{PROJECT}/results/p2_external_dmid_{MASK_SET}'
CODE_DIR = f'{PROJECT}/code'
QSEG_DIR = '/content/drive/MyDrive/QSEG_FINAL/code'      # Paper 1 package, read only
if not os.path.exists(f'{QSEG_DIR}/qseg/quantum_sim.py'):
    hits = glob.glob('/content/drive/MyDrive/**/qseg/quantum_sim.py', recursive=True)
    assert hits, 'Paper 1 qseg package not found on Drive'
    QSEG_DIR = os.path.dirname(os.path.dirname(hits[0]))
for p in (QSEG_DIR, CODE_DIR):
    if p not in sys.path: sys.path.insert(0, p)
assert os.path.exists(f'{CODE_DIR}/qseg2.py'), 'qseg2.py missing (written by notebook 02 cell 2)'
assert os.path.exists(f'{DMID_DIR}/metadata.csv'), 'DMID crops missing (run 03b first)'
os.makedirs(OUT_DIR, exist_ok=True)
print('dmid :', DMID_DIR); print('runs :', RUNS_DIR); print('out  :', OUT_DIR)


In [ ]:
# 2 · CDD-CESM: how many low-energy (DM) images are usable mass segmentation cases?
seg = pd.read_csv(f'{CDD_DIR}/Radiology_hand_drawn_segmentations_v2.csv')
seg['shape'] = seg['region_shape_attributes'].apply(
    lambda s: json.loads(s).get('name', 'none') if isinstance(s, str) and s.strip() else 'none')
seg['image'] = seg['#filename'].str.replace(r'\.[A-Za-z]+$', '', regex=True).str.strip()
seg = seg[seg['image'].str.contains('_DM_')]
reg = seg[seg['shape'] != 'none'].groupby('image').agg(n_regions=('shape', 'size'),
                                                      shapes=('shape', lambda s: ','.join(sorted(set(s)))))
ms = pd.read_excel(f'{CDD_DIR}/Radiology-manual-annotations.xlsx', sheet_name='mass_description')
ms['image'] = ms['Image_name'].astype(str).str.strip()
ms = ms[ms['Type'].astype(str).str.strip() == 'DM'].merge(reg, left_on='image', right_index=True, how='left')
ms['n_regions'] = ms['n_regions'].fillna(0).astype(int)
dens = ms['Mass density/enhancement pattern'].astype(str)
steps = [('DM rows in mass_description', pd.Series(True, index=ms.index)),
         ('+ has a drawn region', ms.n_regions > 0),
         ('+ "Single" mass', ms['Single/Multiple'].astype(str).str.strip() == 'Single'),
         ('+ not a lymph node (IMLN)', ~dens.str.contains('IMLN')),
         ('+ exactly one drawn region', ms.n_regions == 1),
         ('+ region is a polygon (traced border)', ms.shapes == 'polygon')]
keep = pd.Series(True, index=ms.index); rows = []
for name, cond in steps:
    keep &= cond
    rows.append(dict(step=name, images=int(keep.sum()), patients=int(ms.loc[keep, 'Patient_ID'].nunique()),
                     malignant=int((ms.loc[keep, 'Pathology Classification/ Follow up'] == 'Malignant').sum())))
print(pd.DataFrame(rows).to_string(index=False))
elig = ms[keep]
print('\nEligible by shape of region before the polygon filter:')
print(ms[keep | ((ms.n_regions == 1) & (ms['Single/Multiple'].astype(str).str.strip() == 'Single') & ~dens.str.contains('IMLN'))]
      .shapes.value_counts().to_string())
elig[['image', 'Patient_ID', 'View', 'Mass shape', 'Mass margin', 'Pathology Classification/ Follow up']].to_csv(
    f'{CDD_DIR}/cdd_cesm_dm_eligible.csv', index=False)
print('\nSaved list:', f'{CDD_DIR}/cdd_cesm_dm_eligible.csv')


In [ ]:
# 3 · Load DMID crops + CBIS crops (CBIS is needed for the severing bank and the weight check)
import importlib, tensorflow as tf, qseg2; importlib.reload(qseg2)
from qseg.config import Config
cfg = Config(work_dir=RUNS_DIR)
meta = pd.read_csv(f'{DMID_DIR}/metadata.csv')
def _png(p, mask):
    a = tf.image.decode_png(tf.io.read_file(p), channels=1).numpy()
    return (a > 127).astype(np.float32) if mask else a.astype(np.float32) / 255.0
XD = np.stack([_png(f'{DMID_DIR}/images/{r}.png', False) for r in meta.roi_id])
YD = np.stack([_png(f'{DMID_DIR}/masks/{r}.png', True) for r in meta.roi_id])
print('DMID', XD.shape, '| masses:', len(meta), '| images:', meta.img.nunique(),
      '| empty masks:', int((YD.reshape(len(YD), -1).sum(1) == 0).sum()))
print(meta.groupby(['cls', 'sev']).size().unstack(fill_value=0).to_string())
XC, YC, DFC = qseg2.load_cbis(CBIS_DIR)
print('CBIS', XC.shape, '| TF', tf.__version__)


In [ ]:
# 4 · External test of every finished run (resumable)
def finished_runs():
    out = []
    for d in sorted(glob.glob(f'{RUNS_DIR}/*_fold*_seed*')):
        if os.path.exists(f'{d}/summary.json') and os.path.exists(f'{d}/best.weights.h5'):
            out.append(os.path.basename(d))
    return out

def test_run(run):
    dst = f'{OUT_DIR}/{run}'
    if os.path.exists(f'{dst}/summary.json'):
        return json.load(open(f'{dst}/summary.json'))
    t0 = time.time()
    s_train = json.load(open(f'{RUNS_DIR}/{run}/summary.json'))
    name, fold, seed = s_train['model'], int(s_train['fold']), int(s_train['seed'])
    tf.keras.backend.clear_session()
    m = qseg2.build_p2_model(name, (cfg.image_size, cfg.image_size, cfg.in_channels), cfg.n_qubits, cfg.n_q_layers)
    m.load_weights(f'{RUNS_DIR}/{run}/best.weights.h5')
    tr, va, te = qseg2.fold_split(DFC, fold)
    # weight check: re-score the CBIS test fold, must reproduce the training summary
    cbis = qseg2.per_image_scores(YC[te], m.predict(XC[te], batch_size=cfg.batch_size, verbose=0), cfg.threshold)
    per = qseg2.per_image_scores(YD, m.predict(XD, batch_size=cfg.batch_size, verbose=0), cfg.threshold)
    per.insert(0, 'roi_id', meta.roi_id.values)
    for c in ('img', 'cls', 'sev', 'tissue', 'mask_frac'):
        if c in meta: per[c] = meta[c].values
    out = dict(run=run, model=name, fold=fold, seed=seed, mask_set=MASK_SET, n_dmid=len(per),
               dmid_dice=float(per.dice.mean()), dmid_iou=float(per.iou.mean()),
               dmid_hd95_median=float(np.nanmedian(per.hd95)),
               cbis_dice_recomputed=float(cbis.dice.mean()), cbis_dice_training=float(s_train['dice_per_image']))
    if 'sever' in [l.name for l in m.layers]:
        qseg2.set_sever_bank(m, XC[tr], cfg.batch_size, seed)          # same bank recipe as training
        for mode, tag in ((1, 'const'), (2, 'swap')):
            s = qseg2.per_image_scores(YD, qseg2.predict_with_mode(m, XD, mode, cfg.batch_size), cfg.threshold)
            per[f'dice_{tag}'] = s.dice.values
            out[f'dmid_dice_sever_{tag}'] = float(s.dice.mean())
    out['weights_ok'] = abs(out['cbis_dice_recomputed'] - out['cbis_dice_training']) < 2e-3
    out['seconds'] = round(time.time() - t0, 1)
    os.makedirs(dst, exist_ok=True)
    per.to_csv(f'{dst}/per_image.csv', index=False)
    json.dump(out, open(f'{dst}/summary.json', 'w'), indent=2)
    return out

runs = finished_runs()
print(len(runs), 'finished training runs:', runs)
res = []
for run in runs:
    r = test_run(run); res.append(r)
    print(f"{run:22s} DMID Dice {r['dmid_dice']:.4f}"
          + (f"  const {r['dmid_dice_sever_const']:.4f}  swap {r['dmid_dice_sever_swap']:.4f}" if 'dmid_dice_sever_const' in r else '')
          + f"  | CBIS check {r['cbis_dice_recomputed']:.4f} vs {r['cbis_dice_training']:.4f} "
          + ('OK' if r['weights_ok'] else 'MISMATCH'))


In [ ]:
# 5 · Table (mean over finished folds) + by lesion class and pathology
res = pd.DataFrame(res)
res.to_csv(f'{PROJECT}/results/p2_external_dmid_{MASK_SET}_table.csv', index=False)
cols = [c for c in ['dmid_dice', 'dmid_dice_sever_const', 'dmid_dice_sever_swap', 'dmid_hd95_median', 'cbis_dice_training'] if c in res]
tab = res.groupby('model')[cols].agg(['mean', 'std']).round(4); tab['n_folds'] = res.groupby('model').size()
print(tab.reindex([m for m in qseg2.MODELS2 if m in tab.index]).to_string())
print('\nAll weight checks OK:', bool(res.weights_ok.all()))
per_all = pd.concat([pd.read_csv(f'{OUT_DIR}/{r}/per_image.csv').assign(model=r.split('_fold')[0]) for r in res.run])
for c in ('cls', 'sev'):
    if c in per_all:
        print(f'\nDMID Dice by {c}:'); print(per_all.pivot_table(index='model', columns=c, values='dice', aggfunc='mean').round(3).to_string())
